July 1st, 2026

Combine EM results with assertions data - *relative corroboration*

Notebook re-ran and updated as of **July 24, 2026**

*Import packages and modifying data*

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Read in predictions
preds = pd.read_csv("../predictions.csv")
# Read in assertions data
assertions = pd.read_csv("../assertions/assertions_clean.csv")

**Takes 40s to load in because there is so much data**

In [3]:
# Take a subset of columns in preds (probability and unique ids)
# Theoretically, you only need the probability and the unique ids for boosting
matches = preds[['match_probability', 'unique_id_l', 'unique_id_r']]
# Write out matches to CSV for better storage
matches.to_csv('matches.csv', index=False)

In [4]:
# Get all relatives for each mention_id (both directions)
relatives_l = assertions[['subject_id', 'object_id']].rename(columns={'subject_id': 'mention_id', 'object_id': 'relative_id'})
relatives_r = assertions[['object_id', 'subject_id']].rename(columns={'object_id': 'mention_id', 'subject_id': 'relative_id'})
# Combine both directions
all_relatives = pd.concat([relatives_l, relatives_r]).drop_duplicates()
# Build a dict: mention_id -> set of relative ids
relative_sets = all_relatives.groupby('mention_id')['relative_id'].apply(set).to_dict()

*Look at the data*

In [5]:
matches

,match_probability,unique_id_l,unique_id_r
0,0.999702,ALB-CN-1870-10570,ALB-CN-1880-6166
1,0.999702,ALB-CN-1870-12185,ALB-CN-1880-6171
2,0.999702,ALB-CN-1870-14277,ALB-CN-1880-6173
3,0.999702,ALB-CN-1870-18074,ALB-CN-1880-6174
4,0.999702,ALB-CN-1870-14278,ALB-CN-1880-6175
...,...,...,...
28635362,0.963258,ALB-CN-1870-5722,ALB-CN-1880-1742
28635363,0.394037,ALB-CN-1870-5722,ALB-CN-1880-1878
28635364,0.394037,ALB-CN-1870-5722,ALB-CN-1880-3138
28635365,0.394037,ALB-CN-1870-5722,ALB-CN-1880-3184


In [6]:
all_relatives

,mention_id,relative_id
0,ALB-CN-1870-1,ALB-CN-1870-4
1,ALB-CN-1870-14,ALB-CN-1870-16
2,ALB-CN-1870-22,ALB-CN-1870-24
3,ALB-CN-1870-33,ALB-CN-1870-35
4,ALB-CN-1870-33,ALB-CN-1870-39
...,...,...
56076,ALB-CN-1880-19147,ALB-CN-1880-19146
56077,ALB-CN-1880-10085,ALB-CN-1880-10084
56078,ALB-CN-1880-17516,ALB-CN-1880-17515
56079,ALB-CN-1880-3805,ALB-CN-1880-3804


*Get relatives for each left and right record*

In [7]:
# Map relative sets onto matches for each side
matches['relatives_l'] = matches['unique_id_l'].map(relative_sets)
matches['relatives_r'] = matches['unique_id_r'].map(relative_sets)

*Get relative_probs*

`relative_probs` is calculated as follows:
For each potential pair in **matches**:
- Check whether the `unique_id_l` (1870 mention id) has relatives
    - If they don't have relatives, set `relatives_match_probability` as **NA** 
    - If they do have relatives look through each potential combination of `relatives_l` and `relatives_r`:
        - Look through **match_lookup** for the `match_probability` (from the EM model) and:
            - Take the maximum "match" probability for each `relatives_l`
            - Average all the probability values for `relatives_match_probability`

In [8]:
# Build match lookup dict
match_lookup = matches.set_index(['unique_id_l', 'unique_id_r'])['match_probability'].to_dict()

In [9]:
match_lookup

{('ALB-CN-1870-10570', 'ALB-CN-1880-6166'): 0.9997020953520708,
 ('ALB-CN-1870-12185', 'ALB-CN-1880-6171'): 0.9997020953520708,
 ('ALB-CN-1870-14277', 'ALB-CN-1880-6173'): 0.9997020953520708,
 ('ALB-CN-1870-18074', 'ALB-CN-1880-6174'): 0.9997020953520708,
 ('ALB-CN-1870-14278', 'ALB-CN-1880-6175'): 0.9997020953520708,
 ('ALB-CN-1870-17938', 'ALB-CN-1880-6177'): 0.9997020953520708,
 ('ALB-CN-1870-11669', 'ALB-CN-1880-618'): 0.9997020953520708,
 ('ALB-CN-1870-19009', 'ALB-CN-1880-6181'): 0.9997020953520708,
 ('ALB-CN-1870-8867', 'ALB-CN-1880-6182'): 0.9997020953520708,
 ('ALB-CN-1870-7510', 'ALB-CN-1880-6183'): 0.9997020953520708,
 ('ALB-CN-1870-8867', 'ALB-CN-1880-6184'): 0.9997020953520708,
 ('ALB-CN-1870-12545', 'ALB-CN-1880-6185'): 0.9999530711793492,
 ('ALB-CN-1870-14999', 'ALB-CN-1880-6189'): 0.9999530711793492,
 ('ALB-CN-1870-15100', 'ALB-CN-1880-6190'): 0.9997020953520708,
 ('ALB-CN-1870-21610', 'ALB-CN-1880-6197'): 0.9999926088816468,
 ('ALB-CN-1870-14484', 'ALB-CN-1880-6199'): 

In [10]:
# Build a function to get relatives_match_probability
def get_relatives_match_probability(row):
    # Get the two sets of relatives
    relatives_l = row['relatives_l']
    relatives_r = row['relatives_r']
    # If the left side (1870 census) has no relatives, return nan
    if not isinstance(relatives_l, set) or not isinstance(relatives_r, set): # Also nan if no right relatives (nothing to compare)
        return np.nan
    # Create an empty list to hold best match probabilities
    best_probs = []
    # For each 1870 relative of all 1870 relatives
    for rel_l in relatives_l:
        # Get all non-nan match probabilities
        probs = [
            p for rel_r in relatives_r
            if not np.isnan(p := match_lookup.get((rel_l, rel_r), np.nan))
        ]
        # If no valid matches, append nan
        best_probs.append(max(probs) if probs else np.nan)
    # Return mean of best match probabilities (or nan)
    return np.nanmean(best_probs) if best_probs else np.nan

In [11]:
# Apply get_relatives_match_probability to matches
matches['relatives_match_probability'] = matches.apply(get_relatives_match_probability, axis=1)

/var/folders/js/d_qzf0bs4v1gw_dwxd4bhx7h0000gp/T/ipykernel_2262/3885961802.py:21: RuntimeWarning: Mean of empty slice
  return np.nanmean(best_probs) if best_probs else np.nan


In [15]:
# Write matches to CSV w/ relatives_match_probability
matches_subset = matches[['unique_id_l', 'unique_id_r', 'match_probability', 'relatives_match_probability']]
matches_subset.to_csv('boosted_preds.csv', index=False)

In [28]:
# Remove all NAs for relatives_match_probability
matches_no_nas = matches.copy().dropna(subset=['relatives_match_probability'])

In [30]:
# Display matches with relatives_match_probability
matches_no_nas.head(n=20)

,match_probability,unique_id_l,unique_id_r,relatives_l,relatives_r,relatives_match_probability
91,1.000000,ALB-CN-1870-22142,ALB-CN-1880-6437,{ALB-CN-1870-22147},"{ALB-CN-1880-6438, ALB-CN-1880-6442, ALB-CN-18...",0.999999
95,0.999999,ALB-CN-1870-22147,ALB-CN-1880-6441,{ALB-CN-1870-22142},{ALB-CN-1880-6437},1.000000
115,1.000000,ALB-CN-1870-19782,ALB-CN-1880-6472,{ALB-CN-1870-19778},{ALB-CN-1880-6469},1.000000
128,0.999702,ALB-CN-1870-22563,ALB-CN-1880-6511,{ALB-CN-1870-22560},"{ALB-CN-1880-6514, ALB-CN-1880-6512, ALB-CN-18...",0.394037
154,1.000000,ALB-CN-1870-20591,ALB-CN-1880-6607,"{ALB-CN-1870-20596, ALB-CN-1870-20593}","{ALB-CN-1880-6609, ALB-CN-1880-6611, ALB-CN-18...",0.697018
164,0.999993,ALB-CN-1870-21537,ALB-CN-1880-6636,{ALB-CN-1870-21531},{ALB-CN-1880-6634},0.999999
205,0.394037,ALB-CN-1870-15856,ALB-CN-1880-6790,{ALB-CN-1870-15859},"{ALB-CN-1880-6793, ALB-CN-1880-6794, ALB-CN-18...",0.394037
208,0.394037,ALB-CN-1870-15859,ALB-CN-1880-6794,{ALB-CN-1870-15856},{ALB-CN-1880-6790},0.394037
226,0.999999,ALB-CN-1870-22560,ALB-CN-1880-6853,"{ALB-CN-1870-22563, ALB-CN-1870-22565}","{ALB-CN-1880-6855, ALB-CN-1880-6856, ALB-CN-18...",0.999702
227,0.999702,ALB-CN-1870-22565,ALB-CN-1880-6855,{ALB-CN-1870-22560},{ALB-CN-1880-6853},0.999999


In [34]:
# Look at Dabney Johnson
# FIlter for his mention id, sort on match_probability then relatives_match_probability (descending), and display top 10
matches_no_nas[matches_no_nas['unique_id_l'] == 'ALB-CN-1870-1688'].sort_values(['match_probability', 'relatives_match_probability'], ascending=False).head(n=10)

,match_probability,unique_id_l,unique_id_r,relatives_l,relatives_r,relatives_match_probability
20376,1.000000,ALB-CN-1870-1688,ALB-CN-1880-22721,"{ALB-CN-1870-1691, ALB-CN-1870-1694}","{ALB-CN-1880-22722, ALB-CN-1880-22735, ALB-CN-...",0.999851
18871123,0.999993,ALB-CN-1870-1688,ALB-CN-1880-25680,"{ALB-CN-1870-1691, ALB-CN-1870-1694}","{ALB-CN-1880-25683, ALB-CN-1880-25681, ALB-CN-...",0.963258
17084360,0.999953,ALB-CN-1870-1688,ALB-CN-1880-22460,"{ALB-CN-1870-1691, ALB-CN-1870-1694}","{ALB-CN-1880-22464, ALB-CN-1880-22461, ALB-CN-...",0.678647
1134782,0.999953,ALB-CN-1870-1688,ALB-CN-1880-7003,"{ALB-CN-1870-1691, ALB-CN-1870-1694}","{ALB-CN-1880-7009, ALB-CN-1880-7004, ALB-CN-18...",0.394037
1134911,0.999702,ALB-CN-1870-1688,ALB-CN-1880-10250,"{ALB-CN-1870-1691, ALB-CN-1870-1694}","{ALB-CN-1880-10252, ALB-CN-1880-10251, ALB-CN-...",0.963258
15304153,0.999702,ALB-CN-1870-1688,ALB-CN-1880-18656,"{ALB-CN-1870-1691, ALB-CN-1870-1694}","{ALB-CN-1880-18659, ALB-CN-1880-18658, ALB-CN-...",0.963258
20812079,0.999702,ALB-CN-1870-1688,ALB-CN-1880-28345,"{ALB-CN-1870-1691, ALB-CN-1870-1694}","{ALB-CN-1880-28349, ALB-CN-1880-28350, ALB-CN-...",0.963258
4591700,0.999702,ALB-CN-1870-1688,ALB-CN-1880-15526,"{ALB-CN-1870-1691, ALB-CN-1870-1694}","{ALB-CN-1880-15529, ALB-CN-1880-15531, ALB-CN-...",0.884143
6417960,0.999702,ALB-CN-1870-1688,ALB-CN-1880-12983,"{ALB-CN-1870-1691, ALB-CN-1870-1694}","{ALB-CN-1880-12988, ALB-CN-1880-12986, ALB-CN-...",0.884143
24650299,0.999702,ALB-CN-1870-1688,ALB-CN-1880-4538,"{ALB-CN-1870-1691, ALB-CN-1870-1694}","{ALB-CN-1880-4540, ALB-CN-1880-4547, ALB-CN-18...",0.884143
